In [1]:
import pandas as pd
from pathlib import Path

# =============================================================================
# INPUT / OUTPUT
# =============================================================================

INPUT_CSV = Path("data") / "MASTER_VARIABLES.csv"

DISTRICT_OUTPUT = Path("data") / "government_response_district.csv"
BLOCK_OUTPUT = Path("data") / "government_response_block.csv"

df = pd.read_csv(INPUT_CSV)

print("Input shape:", df.shape)

# =============================================================================
# CLEAN KEYS
# =============================================================================

# "district" lags district splits (e.g. Bajali/Tamulpur still show under their
# old parent district) while "dtname" reflects the current taxonomy used by
# hazard/exposure/vulnerability. Use dtname so all pillars agree on district.
df["district"] = df["dtname"].astype(str).str.strip()
df["timeperiod"] = df["timeperiod"].astype(str).str.strip()

# =============================================================================
# PARAMETERS
# =============================================================================

FISCAL_YEAR_START_MONTH = 4

# =============================================================================
# Z-SCORE
# =============================================================================

def zscore(x):
    std = x.std(ddof=0)
    if std == 0:
        return pd.Series(0, index=x.index)
    return (x - x.mean()) / std

# =============================================================================
# CLASSIFICATION (INVERTED)
# =============================================================================

# Cumulative tender spend is zero-inflated and right-skewed (most districts
# haven't spent much yet in a given month; a few outliers spend a lot), so
# symmetric +-0.5/1.5 sigma cutoffs around the mean never catch the
# below-average majority in the extreme classes - classes 4/5 end up
# unreachable. Use the same one-sided mean/mean+sigma escalation as the
# Himachal Pradesh govtresponse.py reference, which puts the entire
# at-or-below-average cluster (incl. zero spend) into class 5 and reserves
# 1-4 for successively rarer above-average spend.
def classify(z):
    if z <= 0:
        return 5
    elif z <= 1:
        return 4
    elif z <= 2:
        return 3
    elif z <= 3:
        return 2
    else:
        return 1

# =============================================================================
# FINANCIAL YEAR
# =============================================================================

def get_financial_year(tp):

    year, month = map(int, str(tp).split("_"))

    if month >= FISCAL_YEAR_START_MONTH:
        return f"{year}-{year+1}"
    else:
        return f"{year-1}-{year}"

# =============================================================================
# DISTRICT-MONTH TENDER TOTAL
# =============================================================================

district_df = (
    df.groupby(["district", "timeperiod"], as_index=False)
      .agg(
          district_tender_value=("total_tender_awarded_value", "sum")
      )
)

# =============================================================================
# FINANCIAL YEAR
# =============================================================================

district_df["financial_year"] = district_df["timeperiod"].apply(get_financial_year)

district_df["date"] = pd.to_datetime(
    district_df["timeperiod"],
    format="%Y_%m"
)

district_df = district_df.sort_values(
    ["district", "date"]
)

# =============================================================================
# CUMULATIVE TENDER VALUE WITHIN FINANCIAL YEAR
# =============================================================================

district_df["total_tender_awarded_value_fy_cumsum"] = (
    district_df
    .groupby(["district", "financial_year"])["district_tender_value"]
    .cumsum()
)

# =============================================================================
# MONTHWISE Z-SCORE
# =============================================================================

district_df["govtresponse_z"] = (
    district_df.groupby("timeperiod")["total_tender_awarded_value_fy_cumsum"]
    .transform(zscore)
)

# =============================================================================
# GOVERNMENT RESPONSE CLASS
# =============================================================================

district_df["government_response"] = (
    district_df["govtresponse_z"]
    .apply(classify)
)

# =============================================================================
# SAVE DISTRICT OUTPUT
# =============================================================================

district_df.to_csv(DISTRICT_OUTPUT, index=False)

# =============================================================================
# APPEND TO BLOCK DATA
# =============================================================================

block_df = df.copy()

if "government_response" in block_df.columns:
    block_df = block_df.drop(columns=["government_response"])

block_df = block_df.merge(
    district_df[
        [
            "district",
            "timeperiod",
            "district_tender_value",
            "total_tender_awarded_value_fy_cumsum",
            "govtresponse_z",
            "government_response",
        ]
    ],
    on=["district", "timeperiod"],
    how="left",
    validate="many_to_one",
)

# =============================================================================
# SAVE BLOCK OUTPUT
# =============================================================================

block_df.to_csv(BLOCK_OUTPUT, index=False)

# =============================================================================
# SUMMARY
# =============================================================================

print(f"\nDistrict output : {DISTRICT_OUTPUT}")
print(f"Block output    : {BLOCK_OUTPUT}")

print("\nDistrict rows:", len(district_df))
print("Block rows:", len(block_df))

print("\nGovernment Response Distribution")
print(district_df["government_response"].value_counts().sort_index())

print("\nMissing values:")
print(block_df["government_response"].isna().sum())

print("\nDistrict preview:")
print(
    district_df[
        [
            "district",
            "timeperiod",
            "district_tender_value",
            "total_tender_awarded_value_fy_cumsum",
            "govtresponse_z",
            "government_response",
        ]
    ].head()
)

print("\nBlock preview:")
print(
    block_df[
        [
            "object_id",
            "revenue_circle",
            "district",
            "timeperiod",
            "government_response",
        ]
    ].head()
)

Input shape: (11160, 29)

District output : data/government_response_district.csv
Block output    : data/government_response_block.csv

District rows: 2170
Block rows: 11160

Government Response Distribution
government_response
1      64
2      39
3      48
4     169
5    1850
Name: count, dtype: int64

Missing values:
0

District preview:
  district timeperiod  district_tender_value  \
0   BAJALI    2021_04                    0.0   
1   BAJALI    2021_05                    0.0   
2   BAJALI    2021_06                    0.0   
3   BAJALI    2021_07                    0.0   
4   BAJALI    2021_08                    0.0   

   total_tender_awarded_value_fy_cumsum  govtresponse_z  government_response  
0                                   0.0        0.000000                    5  
1                                   0.0        0.000000                    5  
2                                   0.0       -0.325644                    5  
3                                   0.0       -0.2356

      object_id   revenue_circle   district timeperiod  government_response
0  18-300-00101  Gossaigaon (Pt)  KOKRAJHAR    2021_04                    5
1  18-300-00102       Bhowraguri  KOKRAJHAR    2021_04                    5
2  18-300-00103           Dotoma  KOKRAJHAR    2021_04                    5
3  18-300-00104   Kokrajhar (Pt)  KOKRAJHAR    2021_04                    5
4  18-300-00105   Bagribari (Pt)  KOKRAJHAR    2021_04                    5
